In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from kagglehub import KaggleDatasetAdapter

In [2]:
# Seed
random.seed(42)
np.random.seed(42)

In [3]:
# Global variables
CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
OUT_PATH = os.path.join(CURRENT_DIR, 'out')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

# Show info on the terminal about how the execution is going.
VERBOSE = True
# Even if there is a checkpoint, the model is retrained.
FORCE_TRAINING = True
# Name of this experiment that will appear in the result files.
SUBFIX_NAME = 'gradient'

In [7]:
df_data = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "uciml/red-wine-quality-cortez-et-al-2009", 'winequality-red.csv')
df_data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [106]:
df_data['num.co'].nunique()

10

In [ ]:
df_data.describe().T

,count,mean,std,min,25%,50%,75%,max
num.co,8579.0,1.872479,1.342600,0.000000,1.000000,2.000000,3.000000,9.000000e+00
scoma,8579.0,11.968877,24.485079,0.000000,0.000000,0.000000,9.000000,1.000000e+02
charges,8579.0,61253.746227,104014.631836,1212.000000,10123.500000,25694.000000,65924.500000,1.435423e+06
avtisst,8579.0,22.644722,13.211783,1.000000,12.000000,19.666656,31.666656,8.300000e+01
sps,8579.0,25.616218,9.792538,1.199951,19.097656,24.097656,30.199219,9.918750e+01
aps,8579.0,37.790069,19.726791,0.000000,23.000000,35.000000,49.000000,1.430000e+02
surv2m,8579.0,0.635304,0.247696,0.000000,0.507935,0.714966,0.824951,9.669189e-01
surv6m,8579.0,0.520153,0.252976,0.000000,0.344971,0.574951,0.724976,9.439697e-01
hday,8579.0,4.415666,9.177276,1.000000,1.000000,1.000000,3.000000,1.480000e+02
diabetes,8579.0,0.195128,0.396322,0.000000,0.000000,0.000000,0.000000,1.000000e+00


In [82]:
# delete variables with more than «na_threshold» missing values
na_threshold = 800
df_data = df_data.loc[:, df_data.isna().sum() <= na_threshold].copy()
print(df_data.shape)
df_data.isna().sum()

(9105, 27)


sex           0
dzgroup       0
dzclass       0
num.co        0
scoma         1
charges     172
avtisst      82
race         42
sps           1
aps           1
surv2m        1
surv6m        1
hday          0
diabetes      0
dementia      0
ca            0
dnr          30
dnrday       30
meanbp        1
wblc        212
hrt           1
resp          1
temp          1
crea         67
sod           1
adlsc         0
y             0
dtype: int64

In [83]:
# delete rows with missing values
df_data.dropna(inplace=True)
print(df_data.shape)
df_data.isna().sum()

(8579, 27)


sex         0
dzgroup     0
dzclass     0
num.co      0
scoma       0
charges     0
avtisst     0
race        0
sps         0
aps         0
surv2m      0
surv6m      0
hday        0
diabetes    0
dementia    0
ca          0
dnr         0
dnrday      0
meanbp      0
wblc        0
hrt         0
resp        0
temp        0
crea        0
sod         0
adlsc       0
y           0
dtype: int64

In [84]:
# delete not useful categorical variables
df_data.drop(columns=['dzclass', 'ca', 'dnr'], inplace=True)

In [86]:
df_data = pd.get_dummies(df_data, columns=['sex', 'dzgroup', 'race'], drop_first=True)
dummy_columns = df_data.columns[df_data.columns.str.startswith(('sex', 'dzgroup', 'race'))]
df_data[dummy_columns] = df_data[dummy_columns].astype(int)

In [87]:
df_data.shape

(8579, 33)

In [88]:
df_data.nunique()

num.co                    10
scoma                     11
charges                 8193
avtisst                  350
sps                      593
aps                      124
surv2m                   945
surv6m                   933
hday                      84
diabetes                   2
dementia                   2
dnrday                   172
meanbp                   164
wblc                     492
hrt                      184
resp                      66
temp                      98
crea                     129
sod                       60
adlsc                   1614
y                       7007
sex_male                   2
dzgroup_CHF                2
dzgroup_COPD               2
dzgroup_Cirrhosis          2
dzgroup_Colon Cancer       2
dzgroup_Coma               2
dzgroup_Lung Cancer        2
dzgroup_MOSF w/Malig       2
race_black                 2
race_hispanic              2
race_other                 2
race_white                 2
dtype: int64

In [89]:
# assert there is no more categorical variables
columnas_categoricas = df_data.select_dtypes(include=['object', 'category']).columns
columnas_categoricas

Index([], dtype='object')

In [90]:
df_data.to_parquet(
    os.path.join(
        DATA_PATH,
        'tabular',
        'real',
        'support',
        'clean.parquet'
    )
)